In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/naosinlineya/raw-data/2024_Apple.pdf
/kaggle/input/datasets/naosinlineya/raw-data/2024_Tesla.pdf
/kaggle/input/datasets/naosinlineya/raw-data/2024_Microsoft.pdf


# converting pdf to markdown

In [1]:
!pip install pymupdf4llm
!pip install langchain-core langchain-experimental python-dotenv
!pip install langchain-huggingface
!pip install faiss-cpu
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 73.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 35.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 50.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 4.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchai

In [2]:

print("done")
from pathlib import Path
import pymupdf4llm
class PDFtoMarkdwn:
    def convert_pdf(self,pdf_path,output_path:str)-> str:
        pdf_file=Path(pdf_path)
        if not  pdf_file.exists():
            raise FileNotFoundError('pdf file not found')


        output_path=Path(output_path)
        output_path.mkdir(parents=True , exist_ok=True)
        markdown_content=pymupdf4llm.to_markdown(str(pdf_file))
        markdown_file=output_path/ f"{pdf_file.stem}.md"
        markdown_file.write_text(
            markdown_content,
            encoding='utf-8'
        )
        return str(markdown_file)

    def convert_directory(self, input_dir: str, output_dir: str) -> list[str]:
        """
        Convert all PDFs from a directory to Markdown.

        Args:
            input_dir: Directory containing PDF files.
            output_dir: Directory to save markdown files.

        Returns:
            List of generated markdown file paths.
        """
        input_path = Path(input_dir)
        markdown_files = []

        for pdf_file in input_path.glob("*.pdf"):
            markdown_file = self.convert_pdf(
                pdf_path=str(pdf_file),
                output_path=output_dir
            )

            markdown_files.append(markdown_file)

        return markdown_files
    
input_dir='/kaggle/input/datasets/naosinlineya/raw-data/'
output_dir='/kaggle/working'
converter=PDFtoMarkdwn()
markdown_files=converter.convert_directory(
    input_dir=str(input_dir),
    output_dir=str(output_dir)
)
             
        

done

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=6/7.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=6/7.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=9/10.


# semantic chuncking and embedding

In [3]:

def read_markdown(markdown_file: str) -> str:
    """
    Read markdown content.

    Args:
        markdown_file: Markdown file path.

    Returns:
        Markdown content.
    """
    return Path(markdown_file).read_text(encoding="utf-8")

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

markdown_content = read_markdown("/kaggle/working/2024_Apple.md")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.create_documents(
    [markdown_content],
    metadatas=[{"company": "Apple", "year": 2024}]   # attached to every chunk
)

print(len(chunks), max(len(c.page_content) for c in chunks))  # max should be ~1000

vector_store = FAISS.from_documents(chunks, embeddings)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_58/1189532624.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


146 998


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    print("FOLDER:", root)
    for file in files:
        print("   FILE:", file)

# vector store FAISS

In [4]:
company = "Apple"
year = 2024
source_file = "2024_Apple.pdf"

for chunk in chunks:
    chunk.metadata.update({
        "company": company,
        "year": year,
        "source_file": source_file
    })


print(chunks[0].metadata)

{'company': 'Apple', 'year': 2024, 'source_file': '2024_Apple.pdf'}


In [5]:
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

""""

# 1. PDF → Markdown
pdf_path = "/kaggle/input/datasets/naosinlineya/raw-data/2024_Apple.pdf"

markdown = pymupdf4llm.to_markdown(pdf_path)


# 2. Markdown → chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = splitter.create_documents([markdown])


# 3. Chunks → embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

"""
# 4. Embeddings → FAISS
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)


# 5. Ask a question
question = "What was the revenue?"

results = vector_store.similarity_search(question, k=4)


# 6. Show relevant chunks
for i, result in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(result.page_content[:1000])


--- Result 1 ---
||**2024**|**2023**|**2022**|
|---|---|---|---|
|Segment operating income|$ 162,044|$ 150,888|$ 152,895|
|Research and development expense|(31,370)|(29,915)|(26,251)|
|Other corporate expenses, net<br>(1)|(7,458)|(6,672)|(7,207)|
|Total operating income|$ 123,216|$ 114,301|$ 119,437|



(1) Includes general and administrative compensation costs, various nonrecurring charges, and other separately managed costs. 

The following tables show net sales for 2024, 2023 and 2022 and long-lived assets as of September 28, 2024 and September 30, 2023 for countries that individually accounted for 10% or more of the respective totals, as well as aggregate amounts for the remaining countries (in millions): 

||**2024**|**2023**|**2022**|
|---|---|---|---|
|Net sales:||||
|U.S.|$ 142,196|$ 138,573|$ 147,859|
|China<br>(1)|66,952|72,559|74,200|
|Other countries|181,887|172,153|172,269|
|Total net sales|$ 391,035|$ 383,285|$ 394,328|

--- Result 2 ---
Net sales disaggregated by signif

# rag

In [6]:
from pydantic import BaseModel, Field

print("Pydantic is working")

Pydantic is working


In [15]:
class FinancialMetrics(BaseModel):
    revenue: str | int | None = Field(None, alias="Revenue")
    net_income: str | int | None = Field(None, alias="Net Income")
    operating_income: str | int | None = Field(None, alias="Operating Income")
    cash_flow: str | int | None = Field(None, alias="Cash Flow from Operating Activities")
    total_assets: str | int | None = Field(None, alias="Total Assets")
    total_liabilities: str | int | None = Field(None, alias="Total Liabilities")
    risk_factors: str | list | None = Field(None, alias="Top Risk Factors")
    growth_drivers: str | list | None = Field(None, alias="Top Growth Drivers")


In [7]:
class Retriever:
    def __init__(self, vector_store):
        self.vector_store = vector_store

    def invoke(
        self,
        query: str,
        company: str | None = None,
        year: int | None = None,
        top_k: int = 20
    ):
        results = self.vector_store.similarity_search(
            query,
            k=top_k
        )

        # Apply company/year filtering
        if company is not None:
            results = [
                doc for doc in results
                if doc.metadata.get("company") == company
            ]

        if year is not None:
            results = [
                doc for doc in results
                if doc.metadata.get("year") == year
            ]

        return results

In [8]:
retriever = Retriever(vector_store)

documents = retriever.invoke(
    query="income statement revenue net income operating income",
    company="Apple",
    year=2024,
    top_k=20
)

In [9]:
def retrieve_context(
    retriever,
    company: str,
    year: int,
    query: str,
    top_k: int = 8
) -> str:

    documents = retriever.invoke(
        query=query,
        company=company,
        year=year,
        top_k=top_k
    )

    return "\n\n".join(
        doc.page_content
        for doc in documents
    )

In [10]:
context = retrieve_context(
    retriever,
    company="Apple",
    year=2024,
    query="revenue net sales income statement",
    top_k=8
)

print(context[:5000])

The Company evaluates the performance of its reportable segments based on net sales and operating income. Net sales for geographic segments are generally based on the location of customers and sales through the Company’s retail stores located in those geographic locations. Operating income for each segment consists of net sales to third parties, related cost of sales, and operating expenses directly attributable to the segment. The information provided to the Company’s chief operating decision maker for purposes of making decisions and assessing segment performance excludes asset information. 

Apple Inc. | 2024 Form 10-K | 46 

The following table shows information by reportable segment for 2024, 2023 and 2022 (in millions):

Net sales disaggregated by significant products and services for 2024, 2023 and 2022 were as follows (in millions): 

||**2024**|**2023**|**2022**|
|---|---|---|---|
|iPhone|$ 201,183|$ 200,583|$ 205,489|
|Mac|29,984|29,357|40,177|
|iPad|26,694|28,300|29,292|
|We

# financial extraction promt

In [11]:
def build_extraction_prompt(
    company: str,
    year: int,
    context: str
) -> str:

    return f"""
You are an expert financial analyst.

Company: {company}
Fiscal Year: {year}

Use ONLY the information in the context below.

Context:
{context}

Extract the following:

1. Revenue
2. Net Income
3. Operating Income
4. Cash Flow from Operating Activities
5. Total Assets
6. Total Liabilities
7. Top Risk Factors
8. Top Growth Drivers

Rules:

- Use only the provided context.
- Do not invent information.
- Return null if the information is unavailable.
- Financial values must match the report exactly.
- Risk factors should be concise.
- Growth drivers should be concise.


{{
  "Revenue": null,
  "Net Income": null,
  "Operating Income": null,
  "Cash Flow from Operating Activities": null,
  "Total Assets": null,
  "Total Liabilities": null,
  "Top Risk Factors": null,
  "Top Growth Drivers": null
}}
"""




# llm part

In [12]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,   # newer transformers versions call this `dtype`
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# connecting everything

In [13]:
company = "Apple"
year = 2024

query = """
revenue net sales net income operating income
cash flow total assets total liabilities
risk factors growth drivers
"""

context = retrieve_context(
    retriever=retriever,
    company=company,
    year=year,
    query=query
)

prompt = build_extraction_prompt(
    company=company,
    year=year,
    context=context
)
print("chunks used:", len(context.split("\n\n")))
print("context chars:", len(context))
n_tokens = len(generator.tokenizer(prompt)["input_ids"])
print("prompt tokens:", n_tokens)

chunks used: 34
context chars: 6578
prompt tokens: 2826


In [14]:
messages = [{"role": "user", "content": prompt}]
response = generator(
    messages,
    max_new_tokens=500,
    do_sample=False,
    return_full_text=False
)
raw = response[0]["generated_text"]
print(raw)




The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "Revenue": 391035,
  "Net Income": 93736,
  "Operating Income": 123216,
  "Cash Flow from Operating Activities": 118254,
  "Total Assets": 74834,
  "Total Liabilities": 45888,
  "Top Risk Factors": [
    "Regulatory Changes",
    "Supply Chain Disruptions",
    "Competitive Market Dynamics"
  ],
  "Top Growth Drivers": [
    "Strong Demand for High-End Devices",
    "Expanding International Presence",
    "Digital Transformation Initiatives"
  ]
}
```


In [17]:
import re,json
match = re.search(r"\{.*\}", raw, re.DOTALL)
data = json.loads(match.group(0))

# databse

In [16]:

from sqlalchemy import create_engine, text

def get_engine():
    return create_engine("sqlite:////kaggle/working/financial.db")

def create_tables() -> None:
    engine = get_engine()
    query = """
    CREATE TABLE IF NOT EXISTS financial_metrics (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        company TEXT,
        year INTEGER,
        revenue INTEGER,
        net_income INTEGER,
        operating_income INTEGER,
        cash_flow INTEGER,
        total_assets INTEGER,
        total_liabilities INTEGER,
        risk_factors TEXT,
        growth_drivers TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """
    with engine.begin() as connection:
        connection.execute(text(query))
    print("financial_metrics table created.")

create_tables()

financial_metrics table created.


In [18]:
def save_metrics(company: str, year: int, metrics: dict) -> None:
    engine = get_engine()
    query = """
    INSERT INTO financial_metrics (
        company, year, revenue, net_income, operating_income, cash_flow,
        total_assets, total_liabilities, risk_factors, growth_drivers
    )
    VALUES (
        :company, :year, :revenue, :net_income, :operating_income, :cash_flow,
        :total_assets, :total_liabilities, :risk_factors, :growth_drivers
    )
    """
    params = {
        "company": company,
        "year": str(year),
        "revenue": metrics.get("Revenue") or metrics.get("revenue"),
        "net_income": metrics.get("Net Income") or metrics.get("net_income"),
        "operating_income": metrics.get("Operating Income") or metrics.get("operating_income"),
        "cash_flow": metrics.get("Cash Flow from Operating Activities") or metrics.get("cash_flow"),
        "total_assets": metrics.get("Total Assets") or metrics.get("total_assets"),
        "total_liabilities": metrics.get("Total Liabilities") or metrics.get("total_liabilities"),
        "risk_factors": "\n".join(metrics.get("Top Risk Factors", []) or metrics.get("risk_factors", [])),
        "growth_drivers": "\n".join(metrics.get("Top Growth Drivers", []) or metrics.get("growth_drivers", [])),
    }
    with engine.begin() as connection:
        connection.execute(text(query), params)
    print(f"Successfully saved metrics for {company} {year}")


def get_metrics():
    engine = get_engine()
    query = """
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY company, year
                   ORDER BY created_at DESC, id DESC
               ) AS rn
        FROM financial_metrics
    ) t
    WHERE rn = 1
    ORDER BY company
    """
    with engine.connect() as connection:
        result = connection.execute(text(query))
        return [dict(row._mapping) for row in result]

In [19]:
print("data" in globals())   # must print True

True


In [20]:
import json
data["Total Assets"] = None        # known wrong from the model, don't save it
data["Total Liabilities"] = None

save_metrics("Apple", 2024, data)
print(get_metrics())



Successfully saved metrics for Apple 2024
[{'id': 1, 'company': 'Apple', 'year': 2024, 'revenue': 391035, 'net_income': 93736, 'operating_income': 123216, 'cash_flow': 118254, 'total_assets': None, 'total_liabilities': None, 'risk_factors': 'Regulatory Changes\nSupply Chain Disruptions\nCompetitive Market Dynamics', 'growth_drivers': 'Strong Demand for High-End Devices\nExpanding International Presence\nDigital Transformation Initiatives', 'created_at': '2026-10-08 17:02:04', 'rn': 1}]


In [21]:
rows = get_metrics()
print(rows[0]["revenue"], rows[0]["total_assets"], rows[0]["risk_factors"][:100])

391035 None Regulatory Changes
Supply Chain Disruptions
Competitive Market Dynamics


# gradio

In [22]:
import gradio as gr
import pandas as pd

COMPANY, YEAR = "Apple", 2024      # your store only has this report
MAX_CONTEXT_TOKENS = 3000           # hard budget so the CUDA crash can't come back

def answer_question(question):
    if not question.strip():
        return "Type a question first.", ""

    docs = retriever.invoke(question, company=COMPANY, year=YEAR, top_k=6)

    # add chunks until the token budget is used up
    kept, used = [], 0
    for d in docs:
        n = len(generator.tokenizer(d.page_content)["input_ids"])
        if used + n > MAX_CONTEXT_TOKENS:
            break
        kept.append(d.page_content)
        used += n

    if not kept:
        return "No relevant text found in the report.", ""

    context = "\n\n---\n\n".join(kept)
    prompt = f"""You are a financial analyst. Answer using ONLY the context below.
If the answer is not in the context, say "Not found in the report."
Do not guess numbers.

Context:
{context}

Question: {question}
Answer:"""

    out = generator(
        [{"role": "user", "content": prompt}],
        max_new_tokens=300,
        do_sample=False,
        return_full_text=False,
    )
    return out[0]["generated_text"].strip(), context   # context = the evidence

def show_metrics():
    rows = get_metrics()
    if not rows:
        return pd.DataFrame({"info": ["No rows saved yet."]})
    return pd.DataFrame(rows).drop(columns=["rn"], errors="ignore")

with gr.Blocks(title="Financial Report Q&A") as demo:
    gr.Markdown("# Financial Report Q&A\nAnswers come from a 1.5B model. Check the evidence box before trusting any number.")
    with gr.Tab("Ask"):
        q = gr.Textbox(label="Question", placeholder="What was total net sales in 2024?")
        btn = gr.Button("Ask")
        ans = gr.Textbox(label="Answer", lines=6)
        evidence = gr.Textbox(label="Retrieved evidence (verify the answer here)", lines=12)
        btn.click(answer_question, q, [ans, evidence])
        q.submit(answer_question, q, [ans, evidence])
    with gr.Tab("Saved metrics"):
        refresh = gr.Button("Refresh")
        table = gr.Dataframe()
        refresh.click(show_metrics, None, table)

demo.launch(share=True, debug=False)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://ec276658de695986b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both